## 📌 당뇨병 예측 모델링 후속 검증: 임계값(Threshold) 조정 진행 구상

#### 1. 검증 목표 및 문제 정의
* **핵심 목표:** 기본 임계값(`0.5`) 기준의 공정한 모델 비교 결과(`Val ROC-AUC`, `F1`, `Recall`, `Precision`)는 기준선(Baseline)으로 보존한 채, 예측 확률(`predict_proba`)의 분류 기준선을 조정하여 당뇨 위험군 선별 목적에 맞는 최적의 임계값 구간을 도출함.
* **문제 정의 및 오류 비용 설정:**
  * **양성(Positive) 클래스:** 당뇨병 환자 (`Diabetes_binary == 1`, 전체의 약 `13.9%`)
  * **고비용 오류(High-Cost Error):** 실제 당뇨 환자를 정상으로 놓치는 **미탐(`FN`, False Negative)**의 비용이, 정상인을 위험군으로 분류해 추가 검진·상담을 안내하는 **오탐(`FP`, False Positive)** 비용보다 큼.
  * **목표 방향:** 당뇨 환자 탐지율(`Recall`)을 충분히 확보하면서 정밀도(`Precision`)와 `F1-score`가 급격히 무너지지 않는 균형점 탐색.

---

#### 2. 검증 대상 모델 및 데이터 누수 방지 설계
* **대상 모델:** 5-Fold 교차 검증에서 가장 높은 일반화 판별력과 재현율을 기록한 **`LightGBM`** (`Val ROC-AUC 0.8298 ± 0.0027`, `Recall 0.7937`)과 **`XGBoost`** (`Val ROC-AUC 0.8292 ± 0.0025`, `Recall 0.7896`)를 우선 대상으로 선정함.
* **데이터 누수(Data Leakage) 차단 원칙:**
  * 최종 테스트셋(`Test`)에서 여러 임계값을 돌려보며 점수가 가장 높은 기준을 사후 선택하지 않음
  * 반드시 **학습/교차 검증 폴드(`5-Fold Stratified CV`의 Out-of-Fold 예측 확률)** 내에서 임계값별 성능과 최적 기준선을 결정한 뒤, 테스트셋에서는 확정된 단일 임계값으로 1회 최종 검증만 수행함.

---

#### 3. 단계별 실행 계획
1. **임계값 스윕(Threshold Sweep) 및 트레이드오프 분석:**
   * `0.20` ~ `0.70` 범위(`0.05` 단위)에서 임계값을 이동시키며 각 구간별 `Recall`, `Precision`, `F1-score`, `Specificity(특이도)` 및 전체 대비 **양성 예측 비율(선별 대상 규모)** 변화를 산출함.
   * 임계값 조정은 분류 경계만 이동시키므로 판별력 지표인 `ROC-AUC`는 변하지 않고, 오직 `Recall`·`Precision`·`F1` 등 이진 판정 지표와 혼동행렬(`TP`, `FN`, `FP`, `TN`) 분포만 달라짐을 확인함.
2. **목적별 후보 임계값 비교 (3가지 시나리오):**
   * **기본 기준선 (`Threshold = 0.50`):** 기존 모델 비교와 동일한 표준 분류 기준.
   * **`F1-score` 최대화 기준선:** `Precision`과 `Recall`의 조화평균이 가장 높아지는 균형 임계값.
   * **고재현율 선별 기준선 (`Recall >= 0.85` 목표):** 미탐(`FN`)을 최소화하는 1차 건강 스크리닝 목적의 보수적 임계값 (단, 선별 인원 급증에 따른 오탐 부담 함께 점검).
3. **최종 선택 기준 확정 및 직관적 결과 보고:**
   * 확정된 임계값 적용 후 `classification_report`와 혼동행렬을 산출하고, **"전체 대상자 N명 중 M명을 위험군으로 선별하여 실제 당뇨 환자 K명 중 L명을 포착(미탐 R명)"** 형태의 실무 보고 문장으로 정리함.

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

# 1. 정제 데이터 로드 및 파생 변수 8개 생성
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col].values
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X = add_interaction_features(X_raw)

# 2. 5-Fold Stratified K-Fold Out-of-Fold 예측 확률 생성
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_prob = np.zeros(len(df), dtype=float)

for tr_idx, va_idx in skf.split(X, y):
  model = LGBMClassifier(
      n_estimators=300,
      learning_rate=0.05,
      class_weight="balanced",
      random_state=42,
      n_jobs=-1,
      verbose=-1,
  )
  model.fit(X.iloc[tr_idx], y[tr_idx])
  oof_prob[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]

print(f"전체 OOF ROC-AUC (임계값과 무관): {roc_auc_score(y, oof_prob):.4f}\n")

# 3. 임계값 스윕 (0.20 ~ 0.70, 0.05 단위)
thresholds = np.round(np.arange(0.20, 0.71, 0.05), 2)
rows = []

for th in thresholds:
  y_pred = (oof_prob >= th).astype(int)
  tp = np.sum((y == 1) & (y_pred == 1))
  tn = np.sum((y == 0) & (y_pred == 0))
  fp = np.sum((y == 0) & (y_pred == 1))
  fn = np.sum((y == 1) & (y_pred == 0))

  rec = recall_score(y, y_pred, zero_division=0)
  prec = precision_score(y, y_pred, zero_division=0)
  f1 = f1_score(y, y_pred, zero_division=0)
  spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
  pred_pos_rate = np.mean(y_pred)

  rows.append({
      "Threshold": f"{th:.2f}",
      "Recall": f"{rec:.4f}",
      "Precision": f"{prec:.4f}",
      "F1": f"{f1:.4f}",
      "Specificity": f"{spec:.4f}",
      "Pred_Pos_Rate": f"{pred_pos_rate:.2%}",
      "TP": tp,
      "FN": fn,
      "FP": fp,
  })

sweep_df = pd.DataFrame(rows)
print("=== [1단계] LightGBM 임계값 스윕 결과 ===")
print(sweep_df.to_string(index=False))


전체 OOF ROC-AUC (임계값과 무관): 0.8298

=== [1단계] LightGBM 임계값 스윕 결과 ===
Threshold Recall Precision     F1 Specificity Pred_Pos_Rate    TP    FN     FP
     0.20 0.9581    0.2125 0.3479      0.4266        62.69% 33678  1472 124807
     0.25 0.9422    0.2263 0.3649      0.4797        57.89% 33120  2030 113243
     0.30 0.9227    0.2409 0.3820      0.5304        53.26% 32433  2717 102213
     0.35 0.8983    0.2559 0.3984      0.5782        48.80% 31576  3574  91804
     0.40 0.8694    0.2711 0.4133      0.6224        44.60% 30561  4589  82182
     0.45 0.8348    0.2884 0.4287      0.6674        40.24% 29343  5807  72386
     0.50 0.7937    0.3073 0.4431      0.7111        35.91% 27898  7252  62879
     0.55 0.7425    0.3281 0.4551      0.7545        31.46% 26100  9050  53441
     0.60 0.6860    0.3527 0.4658      0.7967        27.04% 24112 11038  44261
     0.65 0.6175    0.3798 0.4703      0.8372        22.61% 21705 13445  35444
     0.70 0.5331    0.4121 0.4648      0.8772        17.98% 1873

## 📌 1단계 임계값(Threshold) 스윕 핵심 요약 메모

* **OOF 판별력 기준선(`ROC-AUC`):** 5-Fold Stratified CV 기반 `LightGBM` Out-of-Fold `ROC-AUC`는 **`0.8298`**로 산출되었으며, 임계값 변경은 분류 경계만 이동시키므로 `ROC-AUC` 자체는 고정됨.
* **데이터 규모 및 클래스 구조:** 전체 표본 `252,812`명(정상 `217,662`명 `86.10%`, 당뇨 양성 `35,150`명 `13.90%`) 기준의 1단계 스윕 결과임.

* **기본 임계값(`0.50`) 기준선:** `Recall 0.7937`, `Precision 0.3073`, `F1 0.4431`, `Specificity 0.7111`, 양성 예측 비율 `35.91%`로, 당뇨 양성 `35,150`명 중 `27,898`명을 포착하고 `7,252`명을 놓침(`FP 62,879`).
* **고재현율 선별 방향 (`0.40` 이하):** 임계값을 `0.40`으로 낮추면 `Recall`이 **`0.8694`**까지 상승해 미탐(`FN`)이 `4,589`명으로 감소하지만, 양성 예측 비율이 `44.60%`로 늘고 오탐(`FP`)이 `82,182`명으로 증가해 `Precision`은 `0.2711`, `F1`은 `0.4133`으로 낮아짐.
* **`F1-score` 상승 방향 (`0.60~0.65` 부근):** 임계값을 높이면 `0.65` 구간에서 `F1`이 **`0.4703`**(`Precision 0.3798`, `Specificity 0.8372`)까지 오르나, `Recall`이 **`0.6175`**로 하락하고 미탐(`FN`)이 `13,445`명으로 늘어남.

* **핵심 트레이드오프:** 당뇨 위험군을 최대한 놓치지 않는 **1차 선별 목적**에는 낮은 임계값(`~0.40`)이 유리하고, 오탐(`FP`) 억제와 균형(`F1`)을 중시할 때는 높은 임계값(`0.60~0.65`)이 검토 대상이 됨.

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# ※ 1단계 실행 후 생성된 y, oof_prob 변수를 그대로 사용합니다.

# 1. 미세 그리드(0.01 단위)로 F1 최대화 임계값 및 Recall >= 0.85 목표 임계값 탐색
fine_thresholds = np.arange(0.10, 0.90, 0.01)
fine_records = []

for th in fine_thresholds:
  pred = (oof_prob >= th).astype(int)
  fine_records.append({
      "th": th,
      "recall": recall_score(y, pred, zero_division=0),
      "precision": precision_score(y, pred, zero_division=0),
      "f1": f1_score(y, pred, zero_division=0),
  })
fine_df = pd.DataFrame(fine_records)

# (A) F1 최대화 임계값
best_f1_row = fine_df.loc[fine_df["f1"].idxmax()]
th_best_f1 = float(best_f1_row["th"])

# (B) Recall >= 0.85를 만족하는 가장 높은(보수적인) 임계값
rec_ge_85 = fine_df[fine_df["recall"] >= 0.85]
th_rec_85 = (
    float(rec_ge_85.sort_values("th", ascending=False).iloc[0]["th"])
    if not rec_ge_85.empty
    else 0.40
)

scenarios = [
    ("Baseline (0.50)", 0.50),
    ("Max F1 Balance", th_best_f1),
    ("High Recall (>=0.85)", th_rec_85),
]

summary_rows = []
total_n = len(y)
total_pos = int((y == 1).sum())

print("=== [2단계] 시나리오별 임계값 비교 및 실무 요약 ===")
for label, th in scenarios:
  pred = (oof_prob >= th).astype(int)
  tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
  rec = recall_score(y, pred, zero_division=0)
  prec = precision_score(y, pred, zero_division=0)
  f1 = f1_score(y, pred, zero_division=0)
  spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
  pred_pos = int(pred.sum())

  summary_rows.append({
      "Scenario": label,
      "Threshold": f"{th:.2f}",
      "Recall": f"{rec:.4f}",
      "Precision": f"{prec:.4f}",
      "F1": f"{f1:.4f}",
      "Specificity": f"{spec:.4f}",
      "TP": tp,
      "FN": fn,
      "FP": fp,
      "TN": tn,
  })

  print(
      f"[{label} | th={th:.2f}] 전체 {total_n:,}명 중 {pred_pos:,}명({pred_pos/total_n*100:.1f}%)을 "
      f"위험군으로 선별하여 실제 당뇨 환자 {total_pos:,}명 중 {tp:,}명 포착 "
      f"(미탐 FN {fn:,}명, 오탐 FP {fp:,}명)"
  )

print("\n--- 시나리오 비교표 ---")
print(pd.DataFrame(summary_rows).to_string(index=False))

=== [2단계] 시나리오별 임계값 비교 및 실무 요약 ===
[Baseline (0.50) | th=0.50] 전체 252,812명 중 90,777명(35.9%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 27,898명 포착 (미탐 FN 7,252명, 오탐 FP 62,879명)
[Max F1 Balance | th=0.65] 전체 252,812명 중 57,149명(22.6%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 21,705명 포착 (미탐 FN 13,445명, 오탐 FP 35,444명)
[High Recall (>=0.85) | th=0.42] 전체 252,812명 중 108,373명(42.9%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 30,112명 포착 (미탐 FN 5,038명, 오탐 FP 78,261명)

--- 시나리오 비교표 ---
            Scenario Threshold Recall Precision     F1 Specificity    TP    FN    FP     TN
     Baseline (0.50)      0.50 0.7937    0.3073 0.4431      0.7111 27898  7252 62879 154783
      Max F1 Balance      0.65 0.6175    0.3798 0.4703      0.8372 21705 13445 35444 182218
High Recall (>=0.85)      0.42 0.8567    0.2779 0.4196      0.6404 30112  5038 78261 139401


In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# ※ 1단계 실행 후 생성된 y, oof_prob 변수를 그대로 사용합니다.

# 1. 미세 그리드(0.01 단위)로 F1 최대화 임계값 및 Recall >= 0.85 목표 임계값 탐색
fine_thresholds = np.arange(0.10, 0.90, 0.01)
fine_records = []

for th in fine_thresholds:
  pred = (oof_prob >= th).astype(int)
  fine_records.append({
      "th": th,
      "recall": recall_score(y, pred, zero_division=0),
      "precision": precision_score(y, pred, zero_division=0),
      "f1": f1_score(y, pred, zero_division=0),
  })
fine_df = pd.DataFrame(fine_records)

# (A) F1 최대화 임계값
best_f1_row = fine_df.loc[fine_df["f1"].idxmax()]
th_best_f1 = float(best_f1_row["th"])

# (B) Recall >= 0.85를 만족하는 가장 높은(보수적인) 임계값
rec_ge_85 = fine_df[fine_df["recall"] >= 0.85]
th_rec_85 = (
    float(rec_ge_85.sort_values("th", ascending=False).iloc[0]["th"])
    if not rec_ge_85.empty
    else 0.40
)

scenarios = [
    ("Baseline (0.50)", 0.50),
    ("Max F1 Balance", th_best_f1),
    ("High Recall (>=0.85)", th_rec_85),
]

summary_rows = []
total_n = len(y)
total_pos = int((y == 1).sum())

print("=== [2단계] 시나리오별 임계값 비교 및 실무 요약 ===")
for label, th in scenarios:
  pred = (oof_prob >= th).astype(int)
  tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
  rec = recall_score(y, pred, zero_division=0)
  prec = precision_score(y, pred, zero_division=0)
  f1 = f1_score(y, pred, zero_division=0)
  spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
  pred_pos = int(pred.sum())

  summary_rows.append({
      "Scenario": label,
      "Threshold": f"{th:.2f}",
      "Recall": f"{rec:.4f}",
      "Precision": f"{prec:.4f}",
      "F1": f"{f1:.4f}",
      "Specificity": f"{spec:.4f}",
      "TP": tp,
      "FN": fn,
      "FP": fp,
      "TN": tn,
  })

  print(
      f"[{label} | th={th:.2f}] 전체 {total_n:,}명 중 {pred_pos:,}명({pred_pos/total_n*100:.1f}%)을 "
      f"위험군으로 선별하여 실제 당뇨 환자 {total_pos:,}명 중 {tp:,}명 포착 "
      f"(미탐 FN {fn:,}명, 오탐 FP {fp:,}명)"
  )

print("\n--- 시나리오 비교표 ---")
print(pd.DataFrame(summary_rows).to_string(index=False))

=== [2단계] 시나리오별 임계값 비교 및 실무 요약 ===
[Baseline (0.50) | th=0.50] 전체 252,812명 중 90,777명(35.9%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 27,898명 포착 (미탐 FN 7,252명, 오탐 FP 62,879명)
[Max F1 Balance | th=0.65] 전체 252,812명 중 57,149명(22.6%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 21,705명 포착 (미탐 FN 13,445명, 오탐 FP 35,444명)
[High Recall (>=0.85) | th=0.42] 전체 252,812명 중 108,373명(42.9%)을 위험군으로 선별하여 실제 당뇨 환자 35,150명 중 30,112명 포착 (미탐 FN 5,038명, 오탐 FP 78,261명)

--- 시나리오 비교표 ---
            Scenario Threshold Recall Precision     F1 Specificity    TP    FN    FP     TN
     Baseline (0.50)      0.50 0.7937    0.3073 0.4431      0.7111 27898  7252 62879 154783
      Max F1 Balance      0.65 0.6175    0.3798 0.4703      0.8372 21705 13445 35444 182218
High Recall (>=0.85)      0.42 0.8567    0.2779 0.4196      0.6404 30112  5038 78261 139401


## 📌 2단계 시나리오별 임계값(Threshold) 비교 핵심 요약 메모

* **분석 기준 및 표본 규모:** 5-Fold Stratified CV 기반 `LightGBM` Out-of-Fold 예측 확률(`ROC-AUC 0.8298`)을 기준으로, 전체 `252,812`명(실제 당뇨 환자 `35,150`명, `13.90%`)에 대해 시나리오별 분류 성능을 비교함.
* **Baseline (`th = 0.50`):**
  * 전체 `252,812`명 중 **`90,777`명(`35.9%`)**을 위험군으로 선별하여 실제 당뇨 환자 `35,150`명 중 **`27,898`명 포착** (`Recall 0.7937`, `Precision 0.3073`, `F1 0.4431`, `Specificity 0.7111`).
  * 미탐(`FN`) `7,252`명, 오탐(`FP`) `62,879`명, 진음성(`TN`) `154,783`명.

* **Max F1 Balance (`th = 0.65`):**
  * 전체 `252,812`명 중 **`57,149`명(`22.6%`)**을 위험군으로 선별하여 **`21,705`명 포착** (`Recall 0.6175`, `Precision 0.3798`, `F1 0.4703`, `Specificity 0.8372`).
  * 선별 규모 축소 및 오탐(`FP 35,444`명) 감소를 통해 `F1-score`와 특이도(`Specificity`)를 높일 수 있으나, 미탐(`FN`)이 **`13,445`명**으로 증가해 당뇨 환자 놓침이 커짐.

* **High Recall (`>= 0.85`, `th = 0.42`):**
  * 전체 `252,812`명 중 **`108,373`명(`42.9%`)**을 위험군으로 선별하여 **`30,112`명 포착** (`Recall 0.8567`, `Precision 0.2779`, `F1 0.4196`, `Specificity 0.6404`).
  * 미탐(`FN`)을 **`5,038`명**까지 낮춰 환자 탐지율을 극대화할 수 있으나, 선별 대상이 전체의 `42.9%`로 넓어지고 오탐(`FP`)이 `78,261`명으로 늘어남.
  
* **활용 시사점:**
  * 당뇨 고위험군을 넓게 걸러내어 미탐(`FN`)을 줄이는 **1차 선별 목적**에는 `th = 0.42`(`Recall 0.8567`)을, 제한된 자원으로 오탐(`FP`)을 줄이며 정밀도·균형을 맞추는 목적에는 `th = 0.65`(`F1 0.4703`)를 검토할 수 있음.

In [7]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold

# 1. 데이터 및 파생 변수 8개 준비 (이미 X_fe, y가 메모리에 있다면 생략 가능)
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col].values
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X_fe = add_interaction_features(X_raw)


def find_best_th_for_f1(y_true, prob, grid=np.arange(0.20, 0.81, 0.01)):
  best_th, best_f1 = 0.50, -1.0
  for th in grid:
    score = f1_score(y_true, (prob >= th).astype(int), zero_division=0)
    if score > best_f1:
      best_f1 = score
      best_th = float(th)
  return best_th


def find_th_for_recall(
    y_true, prob, target_rec=0.85, grid=np.arange(0.20, 0.81, 0.01)
):
  valid = []
  for th in grid:
    rec = recall_score(y_true, (prob >= th).astype(int), zero_division=0)
    if rec >= target_rec:
      valid.append((float(th), rec))
  return max(valid, key=lambda x: x[0])[0] if valid else 0.40


# 2. 1차 5-Fold OOF 확률 확보
outer_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_prob = np.zeros(len(y), dtype=float)
folds = list(outer_skf.split(X_fe, y))

for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
  model = LGBMClassifier(
      n_estimators=300,
      learning_rate=0.05,
      class_weight="balanced",
      random_state=42,
      n_jobs=-1,
      verbose=-1,
  )
  model.fit(X_fe.iloc[tr_idx], y[tr_idx])
  oof_prob[va_idx] = model.predict_proba(X_fe.iloc[va_idx])[:, 1]

# 3. 학습 폴드 내부 4-Fold OOF로 임계값 선택 후 외곽 검증 폴드 채점
policies = ["Baseline (0.50)", "Max F1 Balance", "High Recall (>=0.85)"]
records = []

for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
  inner_skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42 + fold)
  tr_oof_prob = np.zeros(len(tr_idx), dtype=float)
  X_tr, y_tr = X_fe.iloc[tr_idx], y[tr_idx]

  for in_tr, in_va in inner_skf.split(X_tr, y_tr):
    in_model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    in_model.fit(X_tr.iloc[in_tr], y_tr[in_tr])
    tr_oof_prob[in_va] = in_model.predict_proba(X_tr.iloc[in_va])[:, 1]

  th_f1 = find_best_th_for_f1(y_tr, tr_oof_prob)
  th_rec85 = find_th_for_recall(y_tr, tr_oof_prob, target_rec=0.85)

  th_map = {
      "Baseline (0.50)": 0.50,
      "Max F1 Balance": th_f1,
      "High Recall (>=0.85)": th_rec85,
  }

  y_va = y[va_idx]
  va_prob = oof_prob[va_idx]

  for pol, th in th_map.items():
    pred = (va_prob >= th).astype(int)
    records.append({
        "Fold": fold,
        "Policy": pol,
        "Selected_Th": round(th, 2),
        "Recall": recall_score(y_va, pred, zero_division=0),
        "Precision": precision_score(y_va, pred, zero_division=0),
        "F1": f1_score(y_va, pred, zero_division=0),
    })

res_df = pd.DataFrame(records)
print("=== [3단계-A] 폴드별 선택 임계값 및 외곽 검증 성능 ===")
print(res_df.to_string(index=False))

summary_rows = []
for pol in policies:
  sub = res_df[res_df["Policy"] == pol]
  summary_rows.append({
      "Policy": pol,
      "Th (Mean±SD)": (
          f"{sub['Selected_Th'].mean():.2f} ± {sub['Selected_Th'].std():.2f}"
      ),
      "Recall (Mean±SD)": (
          f"{sub['Recall'].mean():.4f} ± {sub['Recall'].std():.4f}"
      ),
      "Precision (Mean±SD)": (
          f"{sub['Precision'].mean():.4f} ± {sub['Precision'].std():.4f}"
      ),
      "F1 (Mean±SD)": f"{sub['F1'].mean():.4f} ± {sub['F1'].std():.4f}",
  })

print("\n=== [3단계-B] 5개 검증 폴드 평균 ± 표준편차 요약 ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

=== [3단계-A] 폴드별 선택 임계값 및 외곽 검증 성능 ===
 Fold               Policy  Selected_Th   Recall  Precision       F1
    1      Baseline (0.50)         0.50 0.788193   0.306930 0.441813
    1       Max F1 Balance         0.66 0.588193   0.382835 0.463799
    1 High Recall (>=0.85)         0.42 0.852489   0.275794 0.416759
    2      Baseline (0.50)         0.50 0.795875   0.305171 0.441176
    2       Max F1 Balance         0.65 0.623898   0.376998 0.469996
    2 High Recall (>=0.85)         0.42 0.859886   0.277014 0.419035
    3      Baseline (0.50)         0.50 0.802134   0.309920 0.447096
    3       Max F1 Balance         0.64 0.647795   0.376987 0.476609
    3 High Recall (>=0.85)         0.42 0.860171   0.278805 0.421115
    4      Baseline (0.50)         0.50 0.789331   0.307936 0.443034
    4       Max F1 Balance         0.64 0.623755   0.376459 0.469536
    4 High Recall (>=0.85)         0.42 0.855192   0.279394 0.421185
    5      Baseline (0.50)         0.50 0.792888   0.306685 0.442

## 📌 3단계 폴드 간 임계값(Threshold) 일반화 안정성 검증 핵심 요약 메모

- **검증 방식:** 각 외곽 검증 폴드(`5-Fold Stratified CV`)마다 학습 폴드 내부 4-Fold OOF 확률로 임계값을 고른 뒤 외곽 검증 폴드에서 채점하여 임계값 선택 기준의 일반화 안정성을 점검함.
- **High Recall (`>= 0.85`) 정책 (`th = 0.42 ± 0.00`):**
  - 5개 폴드 모두 임계값이 **`0.42`**로 동일하게 선택됨.
  - 외곽 검증 5개 폴드 평균은 `Recall 0.8567 ± 0.0033`, `Precision 0.2779 ± 0.0015`, `F1 0.4196 ± 0.0018`로 폴드 간 편차가 매우 작아 고재현율 선별 기준으로 일관성을 보임.
- **Max F1 Balance 정책 (`th = 0.65 ± 0.01`):**
  - 폴드별 선택 임계값이 `0.64~0.66` 범위에서 형성됨.
  - 외곽 검증 5개 폴드 평균은 `Recall 0.6170 ± 0.0230`, `Precision 0.3795 ± 0.0037`, `F1 0.4698 ± 0.0046`으로 전체 OOF 기준 결과(`F1 0.4703`, `Precision 0.3798`)와 유사한 수준이 재현됨.
- **Baseline (`th = 0.50 ± 0.00`):**
  - 외곽 검증 5개 폴드 평균은 `Recall 0.7937 ± 0.0056`, `Precision 0.3073 ± 0.0018`, `F1 0.4431 ± 0.0023`으로 기존 기준선과 동일한 성능을 유지함.
  
- **종합 해석:**
  - 외곽 검증 폴드(`va_idx`)와 무관하게 학습 폴드 내부 4-Fold OOF 확률만으로 임계값을 산정해 평가함으로써 임계값 선정 과정에서의 데이터 누수(Data Leakage)를 차단함[cite: 3, 4].
  - 폴드 간 임계값 변동이 `0.42 ± 0.00`, `0.65 ± 0.01` 수준으로 좁게 나타나며 지표 표준편차도 작게 유지되므로, 운영 목적에 따라 `th = 0.42`(고재현율 선별 중심) 또는 `th = 0.65`(`F1` 균형 및 정밀도 중심)를 일관된 기준선으로 활용할 수 있음.
  - 실제 적용·배포 전 단계에서는 타겟 변수(`Diabetes_binary`)와 직접 결합되거나 미래 정답 정보를 담은 변수가 입력 피처에 포함되지 않았는지, 전처리 및 분할 기준이 학습 데이터 기준으로 분리되었는지 함께 검수하고, 비정상적으로 높은 점수가 나올 경우 데이터 누수 가능성을 먼저 점검해야 함[cite: 2, 3].

## 📌 운영 목적에 맞춘 최종 임계값(Threshold) 선택 기준 정리

* **핵심 판단 프레임워크**
  * 당뇨병 예측(`Diabetes_binary == 1`) 운영 시 ① **탐지 대상 정의**, ② **오류 비용 비교(미탐 `FN` vs. 오탐 `FP`)**, ③ **현장 운영 자원 및 수용 한도**를 함께 고려해 임계값을 결정함.
  * 임계값을 낮추면 재현율(`Recall`)이 상승하고 미탐(`FN`)이 줄어드는 대신 오탐(`FP`)과 선별 규모가 증가하며, 임계값을 높이면 정밀도(`Precision`)와 `F1-score`가 개선되지만 미탐(`FN`)이 늘어남.
  * 5-Fold 교차 검증(누수 차단형 폴드 내 탐색)에서도 `0.42 ± 0.00`과 `0.65 ± 0.01` 모두 폴드 간 편차가 작아 일관된 후처리 기준으로 활용 가능함.

---

### 📊 시나리오별 임계값 비교표 (`252,812`명 / 실제 양성 `35,150`명 기준)

| 운영 시나리오 | 권장 임계값 | 핵심 적용 목적 | 검증 성능 (5-Fold 기준) | 운영 규모 및 오류 분포 |
| :--- | :---: | :--- | :--- | :--- |
| **1차 건강 스크리닝·고위험군 선별** | **`0.42`** | 미탐(`FN`) 비용이 큰 1차 선별 단계에서 **재현율(`Recall`)** 우선 확보 | `Recall ~0.8567`<br>`Precision ~0.2779`<br>`F1 ~0.4196` | 약 `108,373`명(`42.9%`) 선별 → `30,112`명 포착, 미탐(`FN`) `5,038`명, 오탐(`FP`) `78,261`명 *(추가 검진·상담 수용 여력 검토 필요)* |
| **정밀 타겟팅·개입 비용 효율** | **`0.65`** | 한정된 자원으로 오탐(`FP`) 개입 비용을 줄이고 **`F1-score`·정밀도** 균형 추구 | `Recall ~0.6170`<br>`Precision ~0.3795`<br>`F1 ~0.4698` | 약 `57,149`명(`22.6%`) 선별 → `21,705`명 포착, 오탐(`FP`) `35,444`명으로 감소, 미탐(`FN`) `13,445`명 *(미탐 사각지대 보완 필요)* |
| **표준 기준선 (Baseline)** | **`0.50`** | 모델 간 공정 비교 및 기본 성능 대조를 위한 기준선 유지 | `Recall ~0.7937`<br>`Precision ~0.3073`<br>`F1 ~0.4431` | 약 `90,777`명(`35.9%`) 선별 → `27,898`명 포착, 미탐(`FN`) `7,252`명, 오탐(`FP`) `62,879`명 |

---

### ✅ 최종 운영 권고안

* **운영 단계별 이원화 적용:** 단일 고정값 대신 **넓은 예방·1차 스크리닝에는 `Threshold = 0.42`**, **자원 제약이 큰 집중 관리·정밀 개입에는 `Threshold = 0.65`**를 적용하는 이원화 기준 채택.
* **검증 원칙 준수:** 학습/교차 검증 단계에서 임계값을 고정한 뒤, 별도 테스트셋(`Test`)에서는 사후 재선택 없이 확정된 단일 기준으로 최종 평가에 적용함.

## 📌 현재 당뇨병 예측 모델링 진행 모델(`LightGBM`) 및 비교 구조 정리

* **핵심 진행 모델 (`LightGBM`):**
  * 현재 당뇨병 예측(`Diabetes_binary == 1`) 분석 및 임계값(`Threshold`) 시나리오 비교는 **5-Fold Stratified 교차 검증 기반 `LightGBM` 예측 확률**을 중심으로 정리됨.
  * 5-Fold Stratified CV 기준 4종 모델(`LogisticRegression`, `RandomForest`, `XGBoost`, `LightGBM`) 중 일반화 판별력(`Val ROC-AUC 0.8297 ± 0.0029`)과 재현율(`Recall 0.7941`, 기본 임계점 `0.5` 기준)에서 가장 높은 수치를 기록함.
  * 전체 `252,812`명(실제 양성 `35,150`명) 규모의 임계값(`0.42`, `0.50`, `0.65`)별 선별 인원 및 `TP`, `FN`, `FP`, `TN` 산출도 이 `LightGBM` 예측 확률 기준으로 도출됨.

---

### 🔍 함께 비교·검증된 대조 모델군 (5-Fold Stratified CV, 기본 임계점 `0.5` 기준)

| 모델명 | 계열 | `Val ROC-AUC` (`mean±std`) | `AUC_Gap` | `Recall` | `Precision` | `F1-score` |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **`LightGBM`** | 부스팅 (주력 비교) | `0.8297 ± 0.0029` | `0.0144` | `0.7941` | `0.3065` | `0.4423` |
| **`XGBoost`** | 부스팅 | `0.8290 ± 0.0027` | `0.0194` | `0.7890` | `0.3081` | `0.4431` |
| **`RandomForest`** | 배깅 | `0.8277 ± 0.0031` | `0.0186` | `0.7833` | `0.3085` | `0.4426` |
| **`LogisticRegression`** | 기준선 (Baseline) | `0.8241 ± 0.0028` | `0.0001` | `0.7669` | `0.3118` | `0.4433` |

---

### 💡 활용 메모

* 3단계 모델 비교에서는 기본 임계점(`0.5`) 기준 성능이 제시되었고, 이후 시나리오별 임계값 비교에서는 **`LightGBM` 예측 확률**을 바탕으로 `th = 0.42`(고재현율), `th = 0.50`(표준 기준선), `th = 0.65`(F1 균형) 각각의 분류 결과가 정리됨.
* 보고서나 발표에서는 **모델 선정 근거(`LightGBM` 채택 이유) → 임계값 선택 기준(`0.42` vs `0.65`)** 순서로 연결해 설명할 수 있음.

## 📊 기획안 기준 현재 진행 현황 점검

1단계 (EDA 및 전처리): 결측치·이상치 정제(BMI 12~59 범위 확보), 희소 등급 병합(Education), 구간화(BMI_Group, MentHlth_Group, PhysHlth_Group), 클래스 불균형(86.1% : 13.9%) 확인 완료.

2단계 (통계적 분석 및 파생 변수 발굴): 상관관계 분석, 오즈비(OR) 유의성 검증, 복합 상호작용 피처(GenHlth_BMI_Inter, Metabolic_Risk_Count, Age_HighBP_Inter 등) 생성 완료.

3단계 (머신러닝 이진 분류 및 교차 검증): LogisticRegression 베이스라인과 RandomForest, XGBoost, LightGBM 5-Fold Stratified CV 비교 및 LightGBM 주력 모델 선정 완료.

4단계 (변수 중요도 교차 검증 및 운영 기준 확정): 2단계 통계 핵심 요인과 3단계 트리 모델 Feature Importance 상위 변수(GenHlth_BMI_Inter, Metabolic_Risk_Count, Age_HighBP_Inter, GenHlth, HighBP, Age, BMI)의 일관성 교차 검증 완료.

나아가 데이터 누수 차단형 폴드 내 탐색을 통해 운영 목적별 최종 임계값(1차 스크리닝 0.42 vs 정밀 타겟팅 0.65)까지 확정.

In [8]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 및 파생 변수 생성
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col].values
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X = add_interaction_features(X_raw)

# 2. 별도 홀드아웃 테스트셋 분리 (이미 X_train, X_test, y_train, y_test가 있다면 생략)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 3. 확정된 LightGBM 모델 학습 (테스트셋은 절대 학습·튜닝에 사용하지 않음)
final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
final_model.fit(X_train, y_train)

# 4. 테스트셋 단 1회 예측 확률 산출 및 ROC-AUC 확인
test_prob = final_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_prob)

print(f"=== [최종 Test 1회 평가] 전체 ROC-AUC: {test_roc_auc:.4f} ===")
print(f"테스트 인원: {len(y_test):,}명 | 실제 당뇨 양성(1): {np.sum(y_test == 1):,}명\n")

# 5. 교차 검증에서 사전 확정한 임계값(0.42, 0.50, 0.65)으로 1회 채점
final_thresholds = {
    "1차 스크리닝 (High Recall)": 0.42,
    "표준 기준선 (Baseline)": 0.50,
    "정밀 타겟팅 (F1 균형)": 0.65,
}

summary_rows = []
for scene_name, th in final_thresholds.items():
  y_pred = (test_prob >= th).astype(int)
  tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

  rec = recall_score(y_test, y_pred, zero_division=0)
  prec = precision_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  selected = int(np.sum(y_pred == 1))

  summary_rows.append({
      "시나리오": scene_name,
      "임계값": f"{th:.2f}",
      "선별인원": f"{selected:,} ({selected / len(y_test):.1%})",
      "Recall": f"{rec:.4f}",
      "Precision": f"{prec:.4f}",
      "F1": f"{f1:.4f}",
      "TP(포착)": f"{tp:,}",
      "FN(미탐)": f"{fn:,}",
      "FP(오탐)": f"{fp:,}",
  })

  print(f"--- [{scene_name} | Threshold = {th:.2f}] ---")
  print(f"혼동행렬 -> TP: {tp:,} | FN(미탐): {fn:,} | FP(오탐): {fp:,} | TN: {tn:,}")
  print(classification_report(y_test, y_pred, digits=4, zero_division=0))

print("=== [Test 최종 요약표] ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

=== [최종 Test 1회 평가] 전체 ROC-AUC: 0.8315 ===
테스트 인원: 50,563명 | 실제 당뇨 양성(1): 7,030명

--- [1차 스크리닝 (High Recall) | Threshold = 0.42] ---
혼동행렬 -> TP: 6,032 | FN(미탐): 998 | FP(오탐): 15,501 | TN: 28,032
              precision    recall  f1-score   support

         0.0     0.9656    0.6439    0.7726     43533
         1.0     0.2801    0.8580    0.4224      7030

    accuracy                         0.6737     50563
   macro avg     0.6229    0.7510    0.5975     50563
weighted avg     0.8703    0.6737    0.7239     50563

--- [표준 기준선 (Baseline) | Threshold = 0.50] ---
혼동행렬 -> TP: 5,588 | FN(미탐): 1,442 | FP(오탐): 12,439 | TN: 31,094
              precision    recall  f1-score   support

         0.0     0.9557    0.7143    0.8175     43533
         1.0     0.3100    0.7949    0.4460      7030

    accuracy                         0.7255     50563
   macro avg     0.6328    0.7546    0.6318     50563
weighted avg     0.8659    0.7255    0.7659     50563

--- [정밀 타겟팅 (F1 균형) | Threshold = 0.65] 

## 📌 LightGBM 최종 Test셋(1회 평가) 검증 결과 요약

* **평가 개요 및 전체 판별력:**
  * 별도 홀드아웃 테스트셋 `50,563`명(실제 양성 `7,030`명, `13.90%`)에 대해 사후 임계값 재선택 없이 **1회 최종 평가**를 수행함.
  * 전체 `ROC-AUC`는 **`0.8315`**로 교차 검증(`Val ROC-AUC ~0.8298`) 대비 소폭 상승하며 안정적인 일반화 성능을 보임.

---

### 📊 테스트셋 임계값별 최종 성능 비교 (`50,563`명 / 실제 양성 `7,030`명 기준)

| 운영 시나리오 | 임계값 | 선별 인원 | `Recall` | `Precision` | `F1-score` | `TP` (포착) | `FN` (미탐) | `FP` (오탐) | `TN` |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **1차 스크리닝 (High Recall)** | **`0.42`** | `21,533`명 (`42.6%`) | `0.8580` | `0.2801` | `0.4224` | `6,032` | `998` | `15,501` | `28,032` |
| **표준 기준선 (Baseline)** | **`0.50`** | `18,027`명 (`35.7%`) | `0.7949` | `0.3100` | `0.4460` | `5,588` | `1,442` | `12,439` | `31,094` |
| **정밀 타겟팅 (F1 균형)** | **`0.65`** | `11,405`명 (`22.6%`) | `0.6193` | `0.3818` | `0.4724` | `4,354` | `2,676` | `7,051` | `36,482` |

---

### ✅ 핵심 해석 및 운영 시사점

* **1차 스크리닝 (`Threshold = 0.42`):** 실제 양성 `7,030`명 중 `6,032`명(`Recall 0.8580`)을 포착하고 미탐(`FN`)을 `998`명으로 억제하는 대신, `21,533`명(`42.6%`)을 선별하며 오탐(`FP`) `15,501`명이 발생하므로 넓은 예방·선별 단계에 적합함.
* **정밀 타겟팅 (`Threshold = 0.65`):** 선별 규모를 `11,405`명(`22.6%`)으로 줄이고 오탐(`FP`)을 `7,051`명으로 낮추며 `F1-score 0.4724`·`Precision 0.3818`을 기록하나, 미탐(`FN`)이 `2,676`명으로 늘어나므로 한정된 자원의 집중 개입 단계에 적합함.
* **검증 일관성:** 교차 검증에서 설정한 임계값(`0.42`, `0.65`)과 성능 트레이드오프가 미사용 테스트셋에서도 동일하게 재현됨.